# HARP 실제 적재 품질 확인 — 2026-10-01

기록 단위는 활성 snapshot의 `sourceId + externalId`입니다. 시설 연결률은 시설만 분모에 포함합니다. 자동 연결은 장소 동일성의 정답 검증이 아닙니다.

재실행: 이 저장소와 Node 의존성, 마이그레이션된 PostgreSQL/PostGIS, 이미 발행된 관광 자료가 필요합니다. 아래는 읽기 전용 트랜잭션의 `tourism:audit`를 실행하며 자료 수집·발행·삭제를 수행하지 않습니다. `.env` 값은 출력하지 않습니다. 재실행 결과는 당시 DB 상태에 따라 달라집니다.


In [1]:
import json, subprocess, tempfile
from pathlib import Path
from collections import Counter

repo = Path.cwd().resolve()
while not (repo / 'scripts/tourism/audit.ts').exists():
    if repo.parent == repo:
        raise RuntimeError('Book해도 저장소 안에서 실행하세요')
    repo = repo.parent
with tempfile.TemporaryDirectory() as directory:
    output = Path(directory) / 'audit.json'
    result = subprocess.run(['npm', 'run', 'tourism:audit', '--', '--date', '2026-10-01', '--output', str(output)], cwd=repo, capture_output=True, text=True, check=True)
    audit = json.loads(output.read_text())
print(json.dumps({'checkedAt': audit['checkedAt'], 'targetDate': audit['targetDate'], 'counts': audit['counts']}, ensure_ascii=False, indent=2))


{
  "checkedAt": "2026-10-01T07:16:33.429Z",
  "targetDate": "2026-10-01",
  "counts": {
    "catalogPlaces": 20810,
    "snapshots": 3,
    "records": 37
  }
}


In [2]:
# SQL의 $1은 승인·활성 등록부 출처 ID 목록, $2는 기준일입니다.
profile_sql = audit['sql']
print(profile_sql)


SELECT s.id AS "sourceId",v.id AS "snapshotId",
v.content_sha256 AS "contentSha256",v.fetched_at AS "fetchedAt",v.published_at AS "publishedAt",
s.enabled,s.rights_status AS "rightsStatus",v.record_count AS "declaredRecords",
count(r.external_id)::int AS records,
count(*) FILTER (WHERE r.kind='place')::int AS facilities,
count(*) FILTER (WHERE r.kind='event')::int AS events,
count(*) FILTER (WHERE r.kind='place' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "facilitiesWithCoordinates",
count(*) FILTER (WHERE r.kind='place' AND r.canonical_place_id IS NOT NULL)::int AS "linkedFacilities",
count(*) FILTER (WHERE r.kind='event' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "eventsWithCoordinates",
count(*) FILTER (WHERE r.kind='event' AND r.date_status IN ('confirmed','tentative') AND r.start_date<=$2::date AND r.end_date>=$2::date AND r.withdrawn_at IS NULL AND s.enabled AND s.rights_status='approved' AND v.fetched_at>=now()-interval '90 days' AND (r.v

In [3]:
profile = audit['profile']
assert len({row['sourceId'] for row in profile}) == len(profile)
assert all(row['records'] == row['declaredRecords'] for row in profile)
assert all(row['facilities'] + row['events'] == row['records'] for row in profile)
assert all(0 <= row['linkedFacilities'] <= row['facilitiesWithCoordinates'] <= row['facilities'] for row in profile)
facility_count = sum(row['facilities'] for row in profile)
linked_count = sum(row['linkedFacilities'] for row in profile)
rate = 100 * linked_count / facility_count if facility_count else None
print(json.dumps({'facilities': facility_count, 'linkedFacilities': linked_count, 'facilityMatchRatePct': rate, 'events': sum(row['events'] for row in profile), 'eligibleEventsOnDate': sum(row['eligibleEventsOnDate'] for row in profile)}, indent=2))


{
  "facilities": 24,
  "linkedFacilities": 8,
  "facilityMatchRatePct": 33.333333333333336,
  "events": 13,
  "eligibleEventsOnDate": 0
}


In [4]:
diagnostics = audit['diagnostics']
assert len({(row['sourceId'], row['externalId']) for row in diagnostics}) == len(diagnostics)
assert len(diagnostics) == facility_count
reasons = Counter(row['reason'] for row in diagnostics)
assert reasons['LINKED'] == linked_count
print(json.dumps(dict(reasons), ensure_ascii=False, indent=2))
print(json.dumps([{key: row[key] for key in ['sourceId','records','historicalHours','unknownSourceUpdatedAt','eventsWithoutExplicitPeriod','endedEventsBeforeDate','eligibleEventsOnDate']} for row in profile], ensure_ascii=False, indent=2))


{
  "LINKED": 8,
  "NAME_MISMATCH": 10,
  "NO_CATALOG_PLACE_WITHIN_250M": 6
}
[
  {
    "sourceId": "eniwa-events",
    "records": 3,
    "historicalHours": 0,
    "unknownSourceUpdatedAt": 3,
    "eventsWithoutExplicitPeriod": 0,
    "endedEventsBeforeDate": 3,
    "eligibleEventsOnDate": 0
  },
  {
    "sourceId": "furano-events",
    "records": 10,
    "historicalHours": 0,
    "unknownSourceUpdatedAt": 10,
    "eventsWithoutExplicitPeriod": 10,
    "endedEventsBeforeDate": 0,
    "eligibleEventsOnDate": 0
  },
  {
    "sourceId": "furano-places",
    "records": 24,
    "historicalHours": 5,
    "unknownSourceUpdatedAt": 24,
    "eventsWithoutExplicitPeriod": 0,
    "endedEventsBeforeDate": 0,
    "eligibleEventsOnDate": 0
  }
]


## 읽는 범위

8/24(33.3%)은 현재 카탈로그와 보수적 명칭·250m 조건으로 연결된 비율입니다. 미연결은 틀린 장소·폐업을 뜻하지 않으며, 근처 후보도 동일 장소라고 단정하지 않습니다. 명칭 불일치 후보는 수동 대조 대상으로 남깁니다.

행사 13건은 시설 분모에서 제외합니다. 2026-10-01 기준 에니와 3건은 종료됐고, 후라노 10건은 명시적 시작·종료일이 없어 검색 대상 기간을 충족하지 않습니다. `confirmed`는 날짜 기재에 따른 분류이며 실제 개최 보장이 아닙니다.

37건 모두 원문 수정 시점이 없고, 시설 5건은 과거 영업시간을 포함합니다. 수집일을 최신 운영 정보로 해석하지 않습니다. 실제 UI 검증은 별도 `tourism-data-ingestion-20261001.json`에 기록합니다.
